In [2]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os


# 1. Define the custom tool
@tool
def get_weather(city: str) -> str:
    """Get the current weather for a specific city."""
    city_clean = city.lower().strip()
    if "tokyo" in city_clean:
        return "Tokyo is 22°C and sunny ☀️"
    elif "london" in city_clean:
        return "London is 15°C and rainy 🌧️"
    elif "mumbai" in city_clean:
        return "Mumbai is 30°C and humid 🌤️"
    else:
        return f"{city} is 25°C and clear skies ⛅"

tools = [get_weather]
tools_by_name = {tool.name: tool for tool in tools}

# 2. Initialize Gemini LLM
load_dotenv()
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    google_api_key=os.environ["GOOGLE_API_KEY"],
    temperature=0
)

# 3. Attach tools to the LLM
llm_with_tools = llm.bind_tools(tools)

# 4. Ask the question
query = "What is the weather like in Kashmir right now?"
messages = [HumanMessage(content=query)]

# Step A: Model decides which tool to call
ai_msg = llm_with_tools.invoke(messages)
messages.append(ai_msg)

# Step B: Run the tool
for tool_call in ai_msg.tool_calls:
    selected_tool = tools_by_name[tool_call["name"]]
    tool_output = selected_tool.invoke(tool_call["args"])
    # Return tool output back to the model
    messages.append(ToolMessage(content=str(tool_output), tool_call_id=tool_call["id"]))

# Step C: Gemini gives the final answer based on tool result
final_response = llm_with_tools.invoke(messages)

print("\n--- FINAL ANSWER ---")
print(final_response.content)

c:\learning\Agentic AI\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
c:\learning\Agentic AI\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



--- FINAL ANSWER ---
[{'type': 'text', 'text': 'The current weather in Kashmir is 25°C with clear skies.', 'extras': {'signature': 'EroBCrcBARFNMg/2VYFW3Eq9MJ/XI3NLIZkuSqHbnfqiD+Wu+pZsm+1+sITjAnCKCnDE/2jMRecnbuS9XANrzJawwPL6PXYkCbS52nUwDnigQRVRsV90UrSPrtTpl6WliH0ohcM+HhDFG33mQh7xOvY5/l8WFg7bngi2RFYXNwHlrBbTfCApLc37lcig1wkO1jWT2iBSH01WtjVL0ZYcoNNgkHdDfqOF4uo/XhM1yMRyDDntB1dzkaheGzo3'}}]
